# Abschlussprojekt: Der deutsche Immobilienmarkt 2005 bis 2024

## Datengrundlage

Der Datensatz umfasst quartalsweise Kaufpreise je m² für 21 deutsche Städte und drei Immobilientypen (single_family, apartment, multi_family) über den Zeitraum 2005 bis 2024. Der ursprüngliche Datensatz enthielt zusätzliche Spalten (Durchschnitts, Median und Perzentilpreise sowie eine Erschwinglichkeitskennzahl), die sich beim Sichten als fehlerhaft herausstellten. Vermutlich entstand beim Zusammenbau aus mehreren Quellen ein nicht eindeutiger Merge, wodurch sich Zeilen vervielfachten und widersprüchliche Werte je Stadt und Quartal enthielten. Diese Spalten wurden daher aus der Analyse ausgeschlossen. Die verbleibende Kennzahl price_m2 (Transaktionspreis) erwies sich als durchgehend eindeutig und bildet die Grundlage der folgenden Auswertung.

Der Datensatz stammt von Kaggle: "German City-Level Housing Market Dataset"
(https://www.kaggle.com/datasets/meenaaditya/german-city-housing-market-dataset/data),
zusammengestellt aus GREIX-Transaktionspreisen, City-Metrics-Verteilungsdaten und
GREIX-Erschwinglichkeitsindikatoren des Kiel-Instituts.

## Kernthese

Der deutsche Immobilienmarkt ist seit 2005 kein einheitlicher Markt. Die Preise sind nicht nur gestiegen, sondern die Schere zwischen den Städten hat sich zunehmend geöffnet, und Wohnungen haben von diesem Trend stärker profitiert als Einfamilienhäuser.

## Unterthesen

**H1:** Der Gesamtmarkt zeigt einen klaren, sich beschleunigenden Aufwärtstrend, besonders nach 2008.

**H2:** Die Preis Schere zwischen Städten hat sich über die Zeit vergrößert (Divergenz statt Angleichung).

**H3:** Wohnungen (apartment) sind stärker im Preis gestiegen als Einfamilienhäuser (single_family).

In [2]:
import pandas as pd
import plotly.express as px

In [3]:
dataset = pd.read_csv("city_master_dataset.csv")
print(len(dataset))
dataset.head(3)

30019


,city,date,year,quarter,property_type,price_m2,avg_price_m2,med_price_m2,p75_price_m2,p25_price_m2,affordability_ratio
0,Bonn,2005-01-01,2005,1,single_family,2000.0,NaN,NaN,NaN,NaN,0.178156
1,Bonn,2005-01-01,2005,1,single_family,2000.0,NaN,NaN,NaN,NaN,0.222695
2,Bonn,2005-01-01,2005,1,single_family,2000.0,NaN,NaN,NaN,NaN,0.129580


In [4]:
# Reduktion auf die sauberen Kernspalten, Duplikate entfernen.
# Grund: avg_price_m2, med_price_m2, p75/p25 und affordability_ratio enthalten
# widersprüchliche Werte je Stadt/Quartal (vermutlich Merge-Fehler beim Datensatz-Aufbau).
# price_m2 selbst ist sauber, Fokus liegt darauf, der Rest wird ignoriert.
immo = dataset[["city", "date", "year", "quarter", "property_type", "price_m2"]].drop_duplicates()
print(len(immo))
immo.head(3)

3512


,city,date,year,quarter,property_type,price_m2
0,Bonn,2005-01-01,2005,1,single_family,2000.0
3,Bonn,2005-04-01,2005,2,single_family,1900.0
6,Bonn,2005-07-01,2005,3,single_family,1900.0


In [5]:
# Umwandlung von date (Text) in ein echtes Datum
immo["date"] = pd.to_datetime(immo["date"])
immo.dtypes

city                        str
date             datetime64[us]
year                      int64
quarter                   int64
property_type               str
price_m2                float64
dtype: object

In [6]:
# Überblick: Städte, Immobilientypen, Zeitraum
print(immo["city"].nunique())
print(immo["property_type"].unique())
print(immo["date"].min(), "-", immo["date"].max())

21
<StringArray>
['single_family', 'apartment', 'multi_family']
Length: 3, dtype: str
2005-01-01 00:00:00 - 2024-10-01 00:00:00


In [7]:
# Durchschnittlicher Preis je Quartal (1 bis 4), über alle Jahre und Staedte gemittelt
saison_quartal = immo.groupby("quarter")["price_m2"].mean().round(2).reset_index()
saison_quartal

,quarter,price_m2
0,1,2546.47
1,2,2604.90
2,3,2630.07
3,4,2650.91


In [8]:
# Durchschnittlicher Preis je Jahr und Quartal
preis_jahr_quartal = immo.groupby(["year", "quarter"])["price_m2"].mean().round(2).reset_index()
preis_jahr_quartal.head(8)

,year,quarter,price_m2
0,2005,1,1684.62
1,2005,2,1710.26
2,2005,3,1733.33
3,2005,4,1715.38
4,2006,1,1728.21
5,2006,2,1782.05
6,2006,3,1733.33
7,2006,4,1735.90


In [9]:
# Jahresmittelwert eigenständig berechnen (unabhängig von späteren Zellen),
# damit dieser Abschnitt auch beim Neuausführen von oben nach unten funktioniert
jahresmittel = immo.groupby("year")["price_m2"].mean().round(2).reset_index()
jahresmittel = jahresmittel.rename(columns={"price_m2": "jahresmittel"})

saison_vergleich = preis_jahr_quartal.merge(jahresmittel, on="year")
saison_vergleich.head(8)

,year,quarter,price_m2,jahresmittel
0,2005,1,1684.62,1710.90
1,2005,2,1710.26,1710.90
2,2005,3,1733.33,1710.90
3,2005,4,1715.38,1710.90
4,2006,1,1728.21,1744.87
5,2006,2,1782.05,1744.87
6,2006,3,1733.33,1744.87
7,2006,4,1735.90,1744.87


In [10]:
# Prozentuale Abweichung des Quartals vom eigenen Jahresmittel
saison_vergleich["abweichung_prozent"] = ((saison_vergleich["price_m2"] / saison_vergleich["jahresmittel"] - 1) * 100).round(2)
saison_vergleich.head(8)

,year,quarter,price_m2,jahresmittel,abweichung_prozent
0,2005,1,1684.62,1710.90,-1.54
1,2005,2,1710.26,1710.90,-0.04
2,2005,3,1733.33,1710.90,1.31
3,2005,4,1715.38,1710.90,0.26
4,2006,1,1728.21,1744.87,-0.95
5,2006,2,1782.05,1744.87,2.13
6,2006,3,1733.33,1744.87,-0.66
7,2006,4,1735.90,1744.87,-0.51


In [11]:
# Durchschnittliche, trendbereinigte Saisonabweichung je Quartal
saison_vergleich.groupby("quarter")["abweichung_prozent"].mean().round(2)

quarter
1   -2.40
2   -0.10
3    0.89
4    1.61
Name: abweichung_prozent, dtype: float64

## Saisonmuster: Rechtfertigung der Jahresbetrachtung

Da die Rohdaten quartalsweise vorliegen, die folgende Analyse aber überwiegend mit Jahreswerten arbeitet, wird hier geprüft, ob dabei ein wiederkehrendes Saisonmuster verloren geht. Eine einfache Mittelung je Quartal über alle Jahre zeigt einen Anstieg von 2.546,47 € (Q1) auf 2.650,91 € (Q4), dieser Unterschied könnte jedoch allein durch den langfristigen Preisanstieg (siehe H1) verursacht sein und nicht durch echte Saisonalität.

Um das zu prüfen, wurde für jedes Quartal die Abweichung vom jeweiligen Jahresmittelwert berechnet und über alle 20 Jahre gemittelt. Dieser trendbereinigte Vergleich zeigt weiterhin ein klares Muster: Das erste Quartal liegt im Schnitt 2,40 % unter dem Jahresmittel, das vierte Quartal 1,61 % darüber. Die Werte steigen von Q1 zu Q4 gleichmäßig an.

Das bestätigt ein echtes, wenn auch moderates Saisonmuster mit rund 4 Prozentpunkten Spannweite zwischen dem schwächsten und stärksten Quartal. Da diese Schwankung im Vergleich zu den in H1 beschriebenen Jahresveränderungen (teils über 10 Prozentpunkte) klein ausfällt, wird die folgende Analyse auf Jahresbasis fortgesetzt, das Saisonmuster wird jedoch als bekannte Nebenschwankung dokumentiert.

In [12]:
# Durchschnittlicher Preis je Quartal, über alle Staedte und Immobilientypen gemittelt
preis_zeit = immo.groupby("date")["price_m2"].mean().reset_index()
preis_zeit.head()

,date,price_m2
0,2005-01-01,1684.615385
1,2005-04-01,1710.256410
2,2005-07-01,1733.333333
3,2005-10-01,1715.384615
4,2006-01-01,1728.205128


In [49]:
# Trendbereinigte Saisonabweichung je Quartal, praesentationsfertiges Balkendiagramm
saison_ergebnis = saison_vergleich.groupby("quarter")["abweichung_prozent"].mean().round(2).reset_index()

fig = px.bar(saison_ergebnis, x="quarter", y="abweichung_prozent",
             title="Saisonmuster: Abweichung vom Jahresmittel je Quartal",
             labels={"quarter": "Quartal", "abweichung_prozent": "Abweichung vom Jahresmittel (%)"})

fig.update_traces(marker_color="#437066")
fig.update_layout(
    plot_bgcolor="#DCE9E7",
    paper_bgcolor="#DCE9E7",
    width=700, height=450,
    title_x=0.5,
    font=dict(size=14, family="Arial", color="#272D45"),
    margin=dict(l=90, r=40, t=80, b=60),
    bargap=0.7
)
fig.update_xaxes(showgrid=False, color="#272D45", dtick=1)
fig.update_yaxes(showgrid=True, gridcolor="#C5D6D3", ticksuffix=" %", color="#272D45")
fig.show()

In [50]:
# Preisentwicklung ueber die Zeit, gemittelt ueber alle Staedte und Immobilientypen
# Stil und Optik des Diagramms für die Präsentation optimiert

fig = px.line(preis_zeit, x="date", y="price_m2",
              title="Der deutsche Immobilienmarkt: Preisentwicklung 2005-2024",
              labels={"date": "Jahr", "price_m2": "Preis (€/m²)"})

fig.update_traces(line_color="#437066", line_width=3)
fig.update_layout(
    plot_bgcolor="#DCE9E7",
    paper_bgcolor="#DCE9E7",
    width=800, height=500,
    title_x=0.5,
    font=dict(size=14, family="Arial", color="#272D45"),
    margin=dict(l=90, r=40, t=80, b=60)
)
fig.update_xaxes(showgrid=False, color="#272D45")
fig.update_yaxes(showgrid=True, gridcolor="#C5D6D3", tickformat=",.0f", ticksuffix=" €",
                  color="#272D45", range=[900, preis_zeit["price_m2"].max() * 1.05])
fig.update_layout(margin=dict(l=90, r=40, t=80, b=80))

fig.show()

## Preisentwicklung 2005–2024: Ein Markt im Wandel

Der durchschnittliche Kaufpreis je m² ist von rund **1.700 €** (2005) auf bis zu **4.300 €** (Anfang 2022) gestiegen, mehr als eine Verdopplung. Auffällig ist dabei der **Verlauf**, nicht nur die Höhe: Von 2005 bis etwa 2012 wächst der Preis nur langsam und schwankt kaum, ab 2013 beschleunigt sich der Anstieg spürbar und mündet ab 2020 in eine steile Boomphase.

**Der Knick 2022** markiert eine Trendwende: Nach dem Höchststand fällt der Preis innerhalb weniger Quartale um über 10 % — zeitlich passend zur Zinswende der EZB (Leitzinserhöhungen ab Mitte 2022), die Immobilienfinanzierungen deutlich verteuerte und die Nachfrage bremste. Seit 2023 stabilisiert sich der Markt auf niedrigerem Niveau mit ersten Anzeichen einer erneuten Erholung.

**Für H1 bedeutet das:** Der Aufwärtstrend ist real und beschleunigt sich über die Jahre, bricht aber 2022/23 erstmals seit Beginn der Zeitreihe spürbar ein.

In [15]:
# Preis am Anfang und Ende der Gesamtzeitreihe, für Wachstumsraten
# Der price_m2-Wert am Anfang und am Ende jeder Phase wird aus der preis_zeit-Tabelle benötigt

preis_2005 = preis_zeit[preis_zeit["date"] == "2005-01-01"]["price_m2"].iloc[0]
preis_2005

np.float64(1684.6153846153845)

In [16]:
# Preise an den Phasengrenzen, für Wachstumsraten
# Die Gesamtveränderung zwischen zwei Zeitpunkten wird durch die Anzahl der Jahre 
# geteilt, um eine durchschnittliche jährliche Rate zu bekommen (statt nur die 
# reine Gesamtsteigerung zu zeigen).

preis_2012 = preis_zeit[preis_zeit["date"] == "2012-01-01"]["price_m2"].iloc[0]
preis_2013 = preis_zeit[preis_zeit["date"] == "2013-01-01"]["price_m2"].iloc[0]
preis_2019 = preis_zeit[preis_zeit["date"] == "2019-01-01"]["price_m2"].iloc[0]
preis_2020 = preis_zeit[preis_zeit["date"] == "2020-01-01"]["price_m2"].iloc[0]
preis_2024 = preis_zeit[preis_zeit["date"] == "2024-10-01"]["price_m2"].iloc[0]

print(preis_2012, preis_2013, preis_2019, preis_2020, preis_2024)

1927.906976744186 2051.1627906976746 3091.304347826087 3393.478260869565 3741.304347826087


In [17]:
# Jaehrliche Wachstumsrate Phase 1 (2005 bis 2012, 7 Jahre)
wachstum_phase1 = (preis_2012 / preis_2005 - 1) / 7 * 100
wachstum_phase1

np.float64(2.0631380937210824)

In [18]:
# Jaehrliche Wachstumsrate Phase 2 (2013 bis 2019, 6 Jahre)
wachstum_phase2 = (preis_2019 / preis_2013 - 1) / 6 * 100

# Jaehrliche Wachstumsrate Phase 3 (2020 bis Oktober 2024, 4.75 Jahre)
wachstum_phase3 = (preis_2024 / preis_2020 - 1) / 4.75 * 100

print(wachstum_phase1, wachstum_phase2, wachstum_phase3)

2.0631380937210824 8.451641526175687 2.1578610202636646


## Wachstumsraten im Zeitverlauf: Die Beschleunigung im Detail

Der Zeitraum wurde in drei Phasen unterteilt und jeweils die durchschnittliche jährliche Wachstumsrate berechnet. Von 2005 bis 2012 stieg der Preis um 2,06 % pro Jahr, ein moderates Wachstum. Von 2013 bis 2019 beschleunigte sich der Anstieg deutlich auf 8,45 % pro Jahr, mehr als das Vierfache der ersten Phase.

Auffällig ist Phase 3 (2020 bis Oktober 2024). Hier liegt die Rate mit 2,16 % nur knapp über dem Niveau der ersten Phase, obwohl das Diagramm hier die stärkste Boomphase der Zeitreihe zeigt. Der Grund: Phase 3 umfasst sowohl den steilen Anstieg bis Anfang 2022 als auch die Korrektur nach der Zinswende. Beide Bewegungen gleichen sich im Durchschnitt aus, wodurch die reine Wachstumsrate die eigentliche Dynamik verschleiert.

Für H1 bedeutet das: Die Beschleunigung ist real, konzentriert sich aber vor allem auf 2013 bis 2019. Eine Wachstumsrate über längere Zeiträume kann Boom und Korrektur verdecken, was den vollständigen Zeitverlauf im Liniendiagramm umso wichtiger macht.

In [19]:
# Preis am Hoehepunkt (2022-01-01), fuer die Aufteilung von Phase 3
preis_2022 = preis_zeit[preis_zeit["date"] == "2022-01-01"]["price_m2"].iloc[0]
preis_2022

np.float64(4302.173913043478)

In [20]:
# Jaehrliche Wachstumsrate Phase 3a (2020 bis 2022, Boom, 2 Jahre)
wachstum_phase3a = (preis_2022 / preis_2020 - 1) / 2 * 100

# Jaehrliche Wachstumsrate Phase 3b (2022 bis Oktober 2024, Korrektur, 2.75 Jahre)
wachstum_phase3b = (preis_2024 / preis_2022 - 1) / 2.75 * 100

print(wachstum_phase3a, wachstum_phase3b)

13.388853299167202 -4.740686297027881


## Robustheitscheck: Wachstumsrate je Kalenderjahr

Die bisherige Phaseneinteilung (2005 bis 2012, 2013 bis 2019, 2020 bis 2022, 2022 bis 2024) wurde anhand sichtbarer Wendepunkte im Liniendiagramm gewählt. Diese Grenzen sind nicht objektiv hergeleitet und könnten das Ergebnis verzerren, wenn andere Jahre als Grenzen gewählt worden wären.

Um das zu prüfen, wird im nächsten Schritt die Wachstumsrate für jedes einzelne Kalenderjahr berechnet. Das zeigt genauer, ob sich der Trend tatsächlich um 2012/13 verändert oder ob der Übergang fließender verläuft, und macht die bisherige Phaseneinteilung nachvollziehbar oder korrigiert sie bei Bedarf.

In [21]:
# Durchschnittlicher Preis je Kalenderjahr, für den Robustheitscheck
preis_jahr = immo.groupby("year")["price_m2"].mean().round(2).reset_index()
preis_jahr

,year,price_m2
0,2005,1710.90
1,2006,1744.87
2,2007,1722.50
3,2008,1704.27
4,2009,1749.39
5,2010,1794.77
6,2011,1874.42
7,2012,1994.19
8,2013,2105.81
9,2014,2171.74


In [22]:
# Prozentuale Veränderung zum jeweiligen Vorjahr
preis_jahr["wachstum_prozent"] = (preis_jahr["price_m2"].pct_change() * 100).round(2)
preis_jahr

,year,price_m2,wachstum_prozent
0,2005,1710.90,NaN
1,2006,1744.87,1.99
2,2007,1722.50,-1.28
3,2008,1704.27,-1.06
4,2009,1749.39,2.65
5,2010,1794.77,2.59
6,2011,1874.42,4.44
7,2012,1994.19,6.39
8,2013,2105.81,5.60
9,2014,2171.74,3.13


## Ergebnis des Robustheitschecks

Die jaehrliche Betrachtung zeigt ein präziseres Bild als die ursprüngliche Phaseneinteilung. Der Übergang von langsamem zu schnellem Wachstum verläuft fliessender als angenommen und beginnt bereits ab 2009, nicht erst 2012/13. Die vermeintlich stabile Phase 2013 bis 2019 enthaelt zudem einen deutlichen Rücksetzer im Jahr 2014.

Der Hoehepunkt der Zeitreihe liegt mit 13,30 % im Jahr 2021, nicht 2022 wie zuvor angenommen. 2022 zeigt mit 4,27 % bereits eine spürbare Abschwächung. Der Preisrückgang konzentriert sich fast vollständig auf ein einziges Jahr: 2023 mit minus 12,27 %, während 2024 mit minus 0,76 % nahezu stabil verläuft.

Die grobe Phaseneinteilung bildete den Gesamttrend richtig ab, verschleierte aber wichtige Details wie den frühen Wachstumsbeginn, den Rücksetzer 2014 und die Konzentration des Preisrückgangs auf ein einzelnes Jahr.

In [51]:
# Jaehrliche Wachstumsrate als Balkendiagramm, praesentationsfertige Version
fig = px.bar(preis_jahr, x="year", y="wachstum_prozent",
             title="Jährliche Wachstumsrate des Immobilienmarktes 2005 bis 2024",
             labels={"year": "Jahr", "wachstum_prozent": "Wachstum (%)"})

fig.update_traces(marker_color="#437066")
fig.update_layout(
    plot_bgcolor="#DCE9E7",
    paper_bgcolor="#DCE9E7",
    width=800, height=500,
    title_x=0.5,
    font=dict(size=14, family="Arial", color="#272D45"),
    margin=dict(l=90, r=40, t=80, b=80),
    bargap=0.5
)
fig.update_xaxes(showgrid=False, color="#272D45", dtick=1, tickangle=-45)
fig.update_yaxes(showgrid=True, gridcolor="#C5D6D3", ticksuffix=" %", color="#272D45")
fig.show()

## Überarbeitete Phaseneinteilung

Die ursprüngliche Einteilung in drei Phasen wurde anhand sichtbarer Wendepunkte im Liniendiagramm gewählt. Der Robustheitscheck mit den jährlichen Wachstumsraten zeigte, dass diese Grenzen zu grob waren und wichtige Details verdeckten.

Auf Basis der jährlichen Werte wurden die Jahre stattdessen nach ähnlicher Wachstumsstärke gruppiert. Daraus ergeben sich vier statt drei Phasen. Aus der ursprünglichen ersten Phase (2005 bis 2012) wurde eine Trennung in eine Vorkrisen und Finanzkrisenphase (2005 bis 2008, Werte um 0 %) und eine Phase der moderaten Erholung (2009 bis 2014, durchgehend positiv zwischen 2,6 % und 6,4 %). Die ursprüngliche zweite Phase verschob sich zu einer längeren, klar abgegrenzten Boomphase von 2015 bis 2021, in der alle sieben Jahre ununterbrochen über 6,9 % lagen. Die ursprüngliche dritte Phase wurde zu einer eigenständigen Korrekturphase ab 2022, da 2021 als eindeutiger Höhepunkt und nicht als Teil der Korrektur zählt.

Die grobe erste Einteilung führte insgesamt in die richtige Richtung, die feinere, an den Jahreswerten orientierte Gruppierung liefert aber ein präziseres und besser begründbares Bild der Marktentwicklung.

In [24]:
# Phase je Jahr zuordnen, basierend auf der überarbeiteten Einteilung
def phase_zuordnen(jahr):
    if jahr <= 2008:
        return "Phase 1: Vorkrise/Finanzkrise (2005-2008)"
    elif jahr <= 2014:
        return "Phase 2: Moderate Erholung (2009-2014)"
    elif jahr <= 2021:
        return "Phase 3: Boom (2015-2021)"
    else:
        return "Phase 4: Korrektur (2022-2024)"

preis_jahr["phase"] = preis_jahr["year"].apply(phase_zuordnen)
preis_jahr

,year,price_m2,wachstum_prozent,phase
0,2005,1710.90,NaN,Phase 1: Vorkrise/Finanzkrise (2005-2008)
1,2006,1744.87,1.99,Phase 1: Vorkrise/Finanzkrise (2005-2008)
2,2007,1722.50,-1.28,Phase 1: Vorkrise/Finanzkrise (2005-2008)
3,2008,1704.27,-1.06,Phase 1: Vorkrise/Finanzkrise (2005-2008)
4,2009,1749.39,2.65,Phase 2: Moderate Erholung (2009-2014)
5,2010,1794.77,2.59,Phase 2: Moderate Erholung (2009-2014)
6,2011,1874.42,4.44,Phase 2: Moderate Erholung (2009-2014)
7,2012,1994.19,6.39,Phase 2: Moderate Erholung (2009-2014)
8,2013,2105.81,5.60,Phase 2: Moderate Erholung (2009-2014)
9,2014,2171.74,3.13,Phase 2: Moderate Erholung (2009-2014)


In [25]:
# Durchschnittliche jährliche Wachstumsrate je überarbeiteter Phase
preis_jahr.groupby("phase")["wachstum_prozent"].mean().round(2)

phase
Phase 1: Vorkrise/Finanzkrise (2005-2008)   -0.12
Phase 2: Moderate Erholung (2009-2014)       4.13
Phase 3: Boom (2015-2021)                    9.14
Phase 4: Korrektur (2022-2024)              -2.92
Name: wachstum_prozent, dtype: float64

## H1: Zusammenfassung und finales Fazit

Die Analyse bestätigt die Kernaussage von H1: Der Immobilienmarkt zeigt seit 2005 einen klaren, sich verstärkenden Aufwärtstrend. Die vier datengetriebenen Phasen zeigen ein eindeutiges Muster. In der Vorkrisen und Finanzkrisenphase (2005 bis 2008) stagnierten die Preise nahezu (minus 0,12 % pro Jahr im Schnitt). Es folgte eine Phase moderater Erholung (2009 bis 2014) mit durchschnittlich 4,13 % Wachstum pro Jahr. Ab 2015 begann eine sieben Jahre andauernde Boomphase mit durchschnittlich 9,14 % Wachstum pro Jahr, in der jedes einzelne Jahr über 6,9 % lag. Diese Phase endete 2022 in einer Korrektur, die sich mit minus 2,92 % pro Jahr im Schnitt vor allem im Einbruchsjahr 2023 (minus 12,27 %) konzentrierte.

Ein erster Blick anhand grob geschätzter Phasengrenzen hätte diese klare Struktur teilweise verdeckt, insbesondere die Länge und Stärke der Boomphase sowie die Konzentration des Einbruchs auf ein einzelnes Jahr. Erst die Einteilung anhand der tatsächlichen jährlichen Wachstumsraten macht das Muster robust nachvollziehbar.

H1 gilt damit als bestätigt: Der Markt zeigt eine reale, über die Zeit zunehmende Dynamik, die sich in einer außergewöhnlich langen und starken Boomphase (2015 bis 2021) entlädt, bevor eine deutliche, aber zeitlich konzentrierte Korrektur folgt.

In [26]:
# Wachstumsraten der Finanzkrisenjahre im Vergleich
preis_jahr[preis_jahr["year"].isin([2007, 2008, 2009])]

,year,price_m2,wachstum_prozent,phase
2,2007,1722.50,-1.28,Phase 1: Vorkrise/Finanzkrise (2005-2008)
3,2008,1704.27,-1.06,Phase 1: Vorkrise/Finanzkrise (2005-2008)
4,2009,1749.39,2.65,Phase 2: Moderate Erholung (2009-2014)


In [27]:
# Wachstumsraten der Zinswende-Jahre im Vergleich
preis_jahr[preis_jahr["year"].isin([2022, 2023])]

,year,price_m2,wachstum_prozent,phase
17,2022,4171.74,4.27,Phase 4: Korrektur (2022-2024)
18,2023,3659.78,-12.27,Phase 4: Korrektur (2022-2024)


In [28]:
# Stärkster Einbruch in beiden Krisen im direkten Vergleich
min_finanzkrise = preis_jahr[preis_jahr["year"].isin([2007, 2008, 2009])]["wachstum_prozent"].min()
min_zinswende = preis_jahr[preis_jahr["year"].isin([2022, 2023])]["wachstum_prozent"].min()

print(min_finanzkrise, min_zinswende)

-1.28 -12.27


## Krisenvergleich: Finanzkrise 2008 versus Zinswende 2022/23

Der stärkste Jahresrückgang während der Finanzkrise lag bei minus 1,28 % (2007), während der stärkste Rückgang während der Zinswende bei minus 12,27 % (2023) lag, fast das Zehnfache. Der Hauptgrund liegt in der Zinsreaktion der Notenbanken: Nach 2008 senkten EZB und Fed die Leitzinsen und machten Finanzierung günstiger, was den deutschen Markt stützte. 2022 erhöhte die EZB die Zinsen dagegen gezielt gegen die Inflation, wodurch Bauzinsen von rund 1 % auf 3,5 bis 4 % stiegen und die Finanzierung direkt verteuerten.

In [29]:
# Streuung der Preise über alle Städte, je Jahr (Standardabweichung)
streuung_jahr = immo.groupby("year")["price_m2"].std().round(2).reset_index()
streuung_jahr

,year,price_m2
0,2005,436.05
1,2006,463.83
2,2007,482.65
3,2008,494.18
4,2009,527.47
5,2010,550.63
6,2011,624.11
7,2012,694.65
8,2013,764.77
9,2014,844.76


In [30]:
# Beide Tabellen vor dem Zusammenführen umbenennen, damit keine Namenskonflikte entstehen
preis_jahr_kopie = preis_jahr[["year", "price_m2"]].rename(columns={"price_m2": "mittelwert"})
streuung_jahr_kopie = streuung_jahr.rename(columns={"price_m2": "streuung"})

# Zusammenführen ueber die gemeinsame Spalte year
streuung_relativ = preis_jahr_kopie.merge(streuung_jahr_kopie, on="year")
streuung_relativ

,year,mittelwert,streuung
0,2005,1710.90,436.05
1,2006,1744.87,463.83
2,2007,1722.50,482.65
3,2008,1704.27,494.18
4,2009,1749.39,527.47
5,2010,1794.77,550.63
6,2011,1874.42,624.11
7,2012,1994.19,694.65
8,2013,2105.81,764.77
9,2014,2171.74,844.76


In [31]:
# Relative Streuung (Variationskoeffizient) in Prozent
streuung_relativ["streuung_relativ_prozent"] = (streuung_relativ["streuung"] / streuung_relativ["mittelwert"] * 100).round(2)
streuung_relativ


,year,mittelwert,streuung,streuung_relativ_prozent
0,2005,1710.90,436.05,25.49
1,2006,1744.87,463.83,26.58
2,2007,1722.50,482.65,28.02
3,2008,1704.27,494.18,29.00
4,2009,1749.39,527.47,30.15
5,2010,1794.77,550.63,30.68
6,2011,1874.42,624.11,33.30
7,2012,1994.19,694.65,34.83
8,2013,2105.81,764.77,36.32
9,2014,2171.74,844.76,38.90


## H2: Streuung der Preise zwischen den Städten

Um zu prüfen, ob die absoluten Preisunterschiede zwischen Städten lediglich mit dem allgemeinen Preisniveau mitgewachsen sind oder tatsächlich überproportional zugenommen haben, wurde die relative Streuung (Standardabweichung geteilt durch den Mittelwert) je Jahr berechnet.

Die relative Streuung stieg von 25,49 % im Jahr 2005 auf einen Höhepunkt von 41,61 % im Jahr 2017, ein deutlicher, über das reine Preiswachstum hinausgehender Anstieg. Ab 2017 flacht der Anstieg jedoch ab und pendelt sich bis 2021 auf einem Niveau um 40 % ein. Seit der Korrektur ab 2022 sinkt die relative Streuung leicht auf zuletzt rund 38 %.

Für H2 bedeutet das: Die Preisschere zwischen den Städten hat sich real vergrößert, nicht nur die absoluten Preise sind gestiegen. Das Wachstum der Unterschiede konzentriert sich jedoch vor allem auf die Jahre 2005 bis 2017 und erreicht danach ein Plateau, anstatt sich unbegrenzt fortzusetzen.

In [32]:
# Durchschnittlicher Preis je Stadt und Jahr
preis_stadt_jahr = immo.groupby(["city", "year"])["price_m2"].mean().round(2).reset_index()
preis_stadt_jahr.head()

,city,year,price_m2
0,Bochum,2005,1291.67
1,Bochum,2006,1291.67
2,Bochum,2007,1208.33
3,Bochum,2008,1183.33
4,Bochum,2009,1216.67


In [33]:
# Preis je Stadt in den beiden Vergleichsjahren
preis_staedte_2005 = preis_stadt_jahr[preis_stadt_jahr["year"] == 2005]
preis_staedte_2024 = preis_stadt_jahr[preis_stadt_jahr["year"] == 2024]

In [34]:
# Beide Tabellen umbenennen, um Namenskonflikte zu vermeiden, dann zusammenfuehren
preis_staedte_2005_kopie = preis_staedte_2005[["city", "price_m2"]].rename(columns={"price_m2": "preis_2005"})
preis_staedte_2024_kopie = preis_staedte_2024[["city", "price_m2"]].rename(columns={"price_m2": "preis_2024"})

veraenderung_staedte = preis_staedte_2005_kopie.merge(preis_staedte_2024_kopie, on="city")
veraenderung_staedte

,city,preis_2005,preis_2024
0,Bochum,1291.67,2150.00
1,Bonn,1687.50,3862.50
2,Chemnitz,1525.00,1475.00
3,Dortmund,1475.00,2687.50
4,Dresden,1537.50,3212.50
5,Duisburg,1237.50,1858.33
6,Düsseldorf,1833.33,4191.67
7,Erfurt,1325.00,2887.50
8,Frankfurt am Main,2025.00,4900.00
9,Hamburg,1750.00,4833.33


In [35]:
# Prozentuale Veraenderung je Stadt zwischen 2005 und 2024
veraenderung_staedte["veraenderung_prozent"] = ((veraenderung_staedte["preis_2024"] / veraenderung_staedte["preis_2005"] - 1) * 100).round(2)
veraenderung_staedte.sort_values("veraenderung_prozent", ascending=False)

,city,preis_2005,preis_2024,veraenderung_prozent
14,München,2850.00,8600.00,201.75
9,Hamburg,1750.00,4833.33,176.19
13,Lübeck,1400.00,3500.00,150.00
8,Frankfurt am Main,2025.00,4900.00,141.98
16,Potsdam,2025.00,4875.00,140.74
18,Wiesbaden,1800.00,4150.00,130.56
12,Köln,1725.00,3966.67,129.95
15,Münster,1825.00,4187.50,129.45
1,Bonn,1687.50,3862.50,128.89
6,Düsseldorf,1833.33,4191.67,128.64


## Hinweis zur Städte-Abdeckung

Der Vergleich zwischen 2005 und 2024 umfasst 19 von 21 Städten. Für Leipzig (Daten ab 2014) und Rhein-Erft-Kreis (Daten ab 2010) liegen für 2005 keine Werte vor, weshalb diese beiden Städte hier nicht einbezogen werden koennen.

In [52]:
# Preisveraenderung je Stadt 2005 bis 2024, sortiert und praesentationsfertig
fig = px.bar(veraenderung_staedte.sort_values("veraenderung_prozent", ascending=False),
             x="city", y="veraenderung_prozent",
             title="Preisveränderung je Stadt: 2005 bis 2024",
             labels={"city": "Stadt", "veraenderung_prozent": "Veränderung (%)"})

fig.update_traces(marker_color="#437066")
fig.update_layout(
    plot_bgcolor="#DCE9E7",
    paper_bgcolor="#DCE9E7",
    width=900, height=550,
    title_x=0.5,
    font=dict(size=13, family="Arial", color="#272D45"),
    margin=dict(l=90, r=40, t=80, b=120),
    bargap=0.5
)
fig.update_xaxes(showgrid=False, color="#272D45", tickangle=-45)
fig.update_yaxes(showgrid=True, gridcolor="#C5D6D3", ticksuffix=" %", color="#272D45")
fig.show()

## H2: Gewinner und Verlierer im Städtevergleich

Der Vergleich der Preisentwicklung zwischen 2005 und 2024 zeigt erhebliche Unterschiede zwischen den Städten. München führt mit einem Anstieg von 201,75 % das Feld an, gefolgt von Hamburg (176,19 %) und Lübeck (150,00 %). Am unteren Ende steht Chemnitz mit minus 3,28 % als einzige Stadt mit einem tatsächlichen Preisrückgang über den gesamten Zeitraum, deutlich abgesetzt von Duisburg (50,17 %) und Bochum (66,45 %), den beiden nächstniedrigeren Werten.

Diese Spannweite von über 200 Prozentpunkten zwischen der stärksten und der schwächsten Stadt bestätigt H2 auf Ebene einzelner Städte. Die Entwicklung verlief nicht gleichmäßig über alle Standorte, sondern konzentrierte sich auf wenige, überwiegend süddeutsche und norddeutsche Großstädte, während einige ostdeutsche und Ruhrgebietsstädte kaum oder gar nicht von der allgemeinen Preisdynamik profitierten.

Hinweis: Der Vergleich umfasst 19 von 21 Städten. Für Leipzig und Rhein-Erft-Kreis liegen für 2005 keine Daten vor.

In [37]:
# Durchschnittlicher Preis je Immobilientyp und Jahr
preis_typ_jahr = immo.groupby(["property_type", "year"])["price_m2"].mean().round(2).reset_index()
preis_typ_jahr.head()

,property_type,year,price_m2
0,apartment,2005,1675.00
1,apartment,2006,1692.11
2,apartment,2007,1715.79
3,apartment,2008,1706.58
4,apartment,2009,1773.68


In [54]:
# Preisentwicklung je Immobilientyp ueber die Zeit, praesentationsfertige Version
# Deutsche Bezeichnungen fuer die Legende anlegen
preis_typ_jahr["immobilientyp_de"] = preis_typ_jahr["property_type"].map({
    "apartment": "Wohnung",
    "single_family": "Einfamilienhaus",
    "multi_family": "Mehrfamilienhaus"
})

fig = px.line(preis_typ_jahr, x="year", y="price_m2", color="immobilientyp_de",
              title="Preisentwicklung nach Immobilientyp 2005 bis 2024",
              labels={"year": "Jahr", "price_m2": "Preis (€/m²)", "immobilientyp_de": "Immobilientyp"},
              color_discrete_map={"Wohnung": "#437066", "Einfamilienhaus": "#7FA399", "Mehrfamilienhaus": "#B79C7A"})

fig.update_traces(line_width=3)
fig.update_layout(
    plot_bgcolor="#DCE9E7",
    paper_bgcolor="#DCE9E7",
    width=800, height=500,
    title_x=0.5,
    font=dict(size=14, family="Arial", color="#272D45"),
    margin=dict(l=90, r=40, t=80, b=80),
    legend_title_text="Immobilientyp"
)
fig.update_xaxes(showgrid=False, color="#272D45", dtick=2)
fig.update_yaxes(showgrid=True, gridcolor="#C5D6D3", ticksuffix=" €", color="#272D45")
fig.show()

In [39]:
# Preis je Immobilientyp in den beiden Vergleichsjahren
preis_typ_2005 = preis_typ_jahr[preis_typ_jahr["year"] == 2005][["property_type", "price_m2"]].rename(columns={"price_m2": "preis_2005"})
preis_typ_2024 = preis_typ_jahr[preis_typ_jahr["year"] == 2024][["property_type", "price_m2"]].rename(columns={"price_m2": "preis_2024"})

veraenderung_typ = preis_typ_2005.merge(preis_typ_2024, on="property_type")
veraenderung_typ

,property_type,preis_2005,preis_2024
0,apartment,1675.00,3898.81
1,multi_family,1258.33,2462.50
2,single_family,1953.57,3852.94


In [40]:
# Prozentuale Veränderung je Immobilientyp zwischen 2005 und 2024
veraenderung_typ["veraenderung_prozent"] = ((veraenderung_typ["preis_2024"] / veraenderung_typ["preis_2005"] - 1) * 100).round(2)
veraenderung_typ.sort_values("veraenderung_prozent", ascending=False)

,property_type,preis_2005,preis_2024,veraenderung_prozent
0,apartment,1675.00,3898.81,132.76
2,single_family,1953.57,3852.94,97.23
1,multi_family,1258.33,2462.50,95.70


## H3: Preisentwicklung nach Immobilientyp

Der direkte Vergleich der Preise 2005 zu 2024 zeigt: apartment stieg mit 132,76 % am stärksten, deutlich vor single_family (97,23 %) und multi_family (95,70 %), die nahezu gleichauf liegen. H3 wird damit bestätigt, apartment verzeichnet das stärkste prozentuale Wachstum aller drei Typen.

Im Liniendiagramm wirken apartment und single_family über weite Strecken nahezu deckungsgleich, was zunächst gegen H3 zu sprechen schien. Der Grund liegt in unterschiedlichen Startniveaus: single_family begann 2005 bei 1.953,57 Euro, apartment bei nur 1.675,00 Euro. Da beide Typen 2024 auf einem ähnlichen Preisniveau (rund 3.850 bis 3.900 Euro) liegen, musste apartment prozentual särker wachsen, um aufzuholen. Dieser Unterschied wird erst durch die Betrachtung der relativen Veränderung sichtbar, nicht durch den optischen Vergleich der absoluten Preislinien.

Zusätzlich fällt multi_family auf: Diese Kategorie startete mit 1.258,33 Euro auf deutlich niedrigerem Niveau als die beiden anderen Typen und vollzog insbesondere zwischen 2018 und 2022 einen auffällig steilen Anstieg, möglicherweise ein Hinweis auf verstärktes Interesse von Kapitalanlegern an Mehrfamilienhäusern während der Boomphase.

## Zwischenfazit zu H1 bis H3

Die bisherige Analyse stützt die Kernthese: Der deutsche Immobilienmarkt ist seit 2005 kein einheitlicher Markt. Alle drei Hypothesen wurden durch die Daten gestützt, teilweise mit wichtigen Präzisierungen gegenüber der ursprünglichen Annahme.

H1 zeigte einen klaren, aber nicht gleichmäßigen Aufwärtstrend. Die Dynamik konzentrierte sich auf eine sieben Jahre andauernde Boomphase (2015 bis 2021) mit durchschnittlich 9,14 % Wachstum pro Jahr, gefolgt von einer Korrektur, die sich fast vollständig auf das Jahr 2023 (minus 12,27 %) konzentrierte.

H2 bestätigte, dass sich die Preisschere zwischen den Städten real vergrößert hat, nicht nur weil die absoluten Preise insgesamt gestiegen sind. Die relative Streuung wuchs von 25,49 % (2005) auf über 40 % (2017 bis 2021). München (plus 201,75 %) und Hamburg (plus 176,19 %) profitierten am stärksten, während Chemnitz als einzige Stadt einen Preisrückgang verzeichnete.

H3 bestätigte, dass apartment mit 132,76 % das stärkste prozentuale Wachstum aller drei Immobilientypen zeigte, vor single_family (97,23 %) und multi_family (95,70 %).

Im Zuge der Arbeit an H3 entstand ein Zusatzbefund, der über die ursprünglichen Hypothesen hinausgeht: multi_family verhielt sich über weite Strecken der Zeitreihe anders als apartment und single_family, mit einem deutlich niedrigeren Ausgangsniveau und einem auffällig steilen Anstieg zwischen 2018 und 2022. Dieser Befund wird im folgenden Abschnitt vertieft, bevor im Anschluss die Erschwinglichkeit als ergänzende Kennzahl einbezogen wird. Das abschließende Gesamtfazit folgt am Ende des Notebooks.

In [41]:
# Jährliche Wachstumsrate je Immobilientyp, getrennt berechnet
preis_typ_jahr = preis_typ_jahr.sort_values(["property_type", "year"])
preis_typ_jahr["wachstum_prozent"] = preis_typ_jahr.groupby("property_type")["price_m2"].pct_change() * 100
preis_typ_jahr.head(20)

,property_type,year,price_m2,wachstum_prozent
0,apartment,2005,1675.00,NaN
1,apartment,2006,1692.11,1.021493
2,apartment,2007,1715.79,1.399436
3,apartment,2008,1706.58,-0.536779
4,apartment,2009,1773.68,3.931840
5,apartment,2010,1817.50,2.470570
6,apartment,2011,1890.00,3.988996
7,apartment,2012,2025.00,7.142857
8,apartment,2013,2146.25,5.987654
9,apartment,2014,2285.71,6.497845


In [42]:
# Standardabweichung der jährlichen Wachstumsrate je Immobilientyp (Volatilitaet)
preis_typ_jahr.groupby("property_type")["wachstum_prozent"].std().round(2)

property_type
apartment        5.48
multi_family     7.88
single_family    5.65
Name: wachstum_prozent, dtype: float64

## Vertiefung: Mehrfamilienhäuser als eigener Markttrend

Um die Vermutung zu prüfen, dass multi_family (Mehrfamilienhäuser) stärker von Kapitalanlegern geprägt ist als apartment und single_family, wurde die Volatilität (Standardabweichung der jährlichen Wachstumsrate) je Typ berechnet. multi_family zeigt mit 7,88 eine deutlich höhere Schwankungsbreite als apartment (5,48) und single_family (5,65), die sich untereinander kaum unterscheiden.

Diese höhere Volatilität passt zum Bild eines investorengetriebenen Marktsegments: Während selbstgenutztes Wohneigentum (apartment, single_family) tendenziell stabiler nachgefragt wird, unabhängig von kurzfristigen Renditeerwartungen, reagiert multi_family stärker auf Zinsniveau und Anlagestimmung. Das zeigt sich sowohl im stärksten Einzeljahresanstieg (13,71 % im Jahr 2015) als auch im stärksten Einbruch aller drei Typen (minus 18,49 % im Jahr 2023), passend zur Niedrigzinsphase und der anschließenden Zinswende der EZB.

Der ursprünglich als Randnotiz aufgefallene Unterschied bei H3 erweist sich damit als eigenständiger, gut belegbarer Befund: Der deutsche Immobilienmarkt unterscheidet sich nicht nur nach Ort und Wohnform, sondern auch danach, ob eine Immobilie selbst genutzt oder als Kapitalanlage gekauft wird.

## Bonus: Erschwinglichkeit (Affordability Ratio)

Die Spalte affordability_ratio (Mortgage Cost Ratio, Anteil der Hypothekenkosten am Einkommen) wurde in der Hauptanalyse ausgeschlossen, da sie durch den vermuteten Merge Fehler beim Datensatz Aufbau mehrere widersprüchliche Werte je Stadt und Jahr enthielt. Da die Kennzahl inhaltlich wichtig ist, um Preisentwicklung und tatsächliche Erschwinglichkeit zu unterscheiden, wird sie hier mit einer transparent dokumentierten Bereinigung dennoch einbezogen.

Bereinigungslogik: Da nicht feststellbar ist, welcher der mehreren Werte je Stadt und Jahr korrekt ist, wird der Mittelwert der vorhandenen Duplikate verwendet. Dies ist eine bewusste Vereinfachung und keine Fehlerkorrektur im eigentlichen Sinn.

In [43]:
# Erschwinglichkeit je Stadt und Jahr, bereinigt durch Mittelwertbildung der Duplikate

afford_jahr = dataset.groupby(["city", "year"])["affordability_ratio"].mean().round(4).reset_index()
afford_jahr

,city,year,affordability_ratio
0,Bochum,2005,0.1771
1,Bochum,2006,0.1782
2,Bochum,2007,0.1720
3,Bochum,2008,0.1590
4,Bochum,2009,0.1621
...,...,...,...
401,Wiesbaden,2020,0.2812
402,Wiesbaden,2021,0.3149
403,Wiesbaden,2022,0.3413
404,Wiesbaden,2023,0.3051


In [44]:
afford_jahr["affordability_ratio"].isna().sum()

np.int64(35)

In [45]:
# Durchschnittliche Erschwinglichkeit ueber alle Staedte, je Jahr

afford_gesamt = afford_jahr.groupby("year")["affordability_ratio"].mean().round(4).reset_index()
afford_gesamt

,year,affordability_ratio
0,2005,0.2169
1,2006,0.2185
2,2007,0.2296
3,2008,0.2229
4,2009,0.2279
5,2010,0.2257
6,2011,0.2284
7,2012,0.2269
8,2013,0.2283
9,2014,0.2352


In [55]:
# Erschwinglichkeit ueber die Zeit, praesentationsfertige Version
fig = px.line(afford_gesamt, x="year", y="affordability_ratio",
              title="Finanzierbarkeit von Wohneigentum 2005 bis 2024",
              labels={"year": "Jahr", "affordability_ratio": "Hypothekenkosten-Anteil am Einkommen"})

fig.update_traces(line_color="#437066", line_width=3)
fig.update_layout(
    plot_bgcolor="#DCE9E7",
    paper_bgcolor="#DCE9E7",
    width=800, height=500,
    title_x=0.5,
    font=dict(size=14, family="Arial", color="#272D45"),
    margin=dict(l=90, r=40, t=80, b=80)
)
fig.update_xaxes(showgrid=False, color="#272D45", dtick=2)
fig.update_yaxes(showgrid=True, gridcolor="#C5D6D3", tickformat=".0%", color="#272D45")
fig.show()

## Erschwinglichkeit im Zeitverlauf

Die durchschnittliche Erschwinglichkeit von Wohneigentum, gemessen als Anteil der Hypothekenkosten am Einkommen, verschlechterte sich von 21,69 % im Jahr 2005 auf einen Höchststand von 31,15 % im Jahr 2022, ein Anstieg um mehr als 9 Prozentpunkte. Bis etwa 2013 blieb der Wert nahezu stabil zwischen 22 % und 23 %, bevor sich die Verschlechterung ab 2014 beschleunigte und um 2020 sprunghaft zunahm.

Bemerkenswert ist die Entwicklung nach 2022. Während die reinen Kaufpreise ab 2023 spürbar zurückgingen (H1), bleibt die Erschwinglichkeit mit 28,52 % (2023) und 27,08 % (2024) deutlich schlechter als vor der Boomphase. Wie beim Krisenvergleich bereits gezeigt, wirkte die Zinswende 2022 direkt über die Finanzierungskosten: Sinkende Kaufpreise wurden durch gestiegene Bauzinsen teilweise aufgehoben, wodurch sich Wohneigentum trotz günstigerer Preise nicht im gleichen Maße erschwinglicher entwickelte.

Dieser Befund ergänzt die Kernthese um eine wichtige Nuance: Ein Rückgang der Kaufpreise bedeutet nicht automatisch eine Erholung der Erschwinglichkeit, da Zinskosten einen eigenständigen, teils gegenläufigen Einfluss haben.

In [47]:
# Durchschnittliche Erschwinglichkeit je Stadt, ueber alle Jahre gemittelt

afford_stadt = afford_jahr.groupby("city")["affordability_ratio"].mean().round(4).reset_index()
afford_stadt = afford_stadt.sort_values("affordability_ratio", ascending=False)
afford_stadt

,city,affordability_ratio
8,Frankfurt am Main,0.3661
15,München,0.3395
9,Hamburg,0.3327
17,Potsdam,0.3152
13,Leipzig,0.2852
6,Düsseldorf,0.2667
12,Köln,0.2536
19,Stuttgart,0.2529
20,Wiesbaden,0.2512
14,Lübeck,0.2462


## Städtevergleich der Erschwinglichkeit

Der Vergleich zeigt weitgehende Übereinstimmung mit den Ergebnissen aus H2, aber auch eine wichtige Abweichung. Frankfurt am Main weist mit einem durchschnittlichen Hypothekenkosten-Anteil von 36,61 % die schlechteste Erschwinglichkeit aller Städte auf, vor München (33,95 %) und Hamburg (33,27 %). Am unteren Ende stehen Chemnitz (13,48 %) und Duisburg (14,15 %) als die Städte mit der besten Erschwinglichkeit, was zur schwächsten beziehungsweise sogar rückläufigen Preisentwicklung dieser Städte aus H2 passt.

Auffällig ist Frankfurt am Main: Bei der Preissteigerung zwischen 2005 und 2024 lag die Stadt nur auf Platz 4 (plus 141,98 Prozent), deutlich hinter München (plus 201,75 Prozent). Bei der Erschwinglichkeit liegt Frankfurt dennoch vorn. Das deutet darauf hin, dass hier bereits das Ausgangsniveau der Preise 2005 vergleichsweise hoch war und über den gesamten Zeitraum hoch blieb, nicht nur der Anstieg selbst.

Für Kreis Mettmann und Rhein-Erft-Kreis liegen keine verwertbaren Erschwinglichkeitsdaten vor.

In [56]:
# Erschwinglichkeit je Stadt

afford_stadt_ohne_nan = afford_stadt.dropna()

fig = px.bar(afford_stadt_ohne_nan, x="city", y="affordability_ratio",
             title="Durchschnittliche Finanzierbarkeit je Stadt (2005 bis 2024)",
             labels={"city": "Stadt", "affordability_ratio": "Hypothekenkosten-Anteil am Einkommen"})

fig.update_traces(marker_color="#437066")
fig.update_layout(
    plot_bgcolor="#DCE9E7",
    paper_bgcolor="#DCE9E7",
    width=900, height=550,
    title_x=0.5,
    font=dict(size=13, family="Arial", color="#272D45"),
    margin=dict(l=90, r=40, t=80, b=120),
    bargap=0.5
)
fig.update_xaxes(showgrid=False, color="#272D45", tickangle=-45)
fig.update_yaxes(showgrid=True, gridcolor="#C5D6D3", tickformat=".0%", color="#272D45")
fig.show()

## Gesamtfazit

Der deutsche Immobilienmarkt war zwischen 2005 und 2024 kein einheitlicher Markt, sondern ein Markt mit starker zeitlicher, regionaler und struktureller Ausdifferenzierung. Die drei Hypothesen wurden bestätigt, jedoch jeweils mit wichtigen Präzisierungen gegenüber der ursprünglichen Annahme.

Der Aufwärtstrend aus H1 verlief nicht gleichmäßig, sondern konzentrierte sich auf eine siebenjährige Boomphase (2015 bis 2021) mit durchschnittlich 9,14 % Wachstum pro Jahr, gefolgt von einer Korrektur, die sich fast vollständig im Jahr 2023 entlud. Der direkte Vergleich mit der Finanzkrise 2008 (stärkster Rückgang minus 1,28 %) zeigte, dass die Korrektur 2023 (minus 12,27 %) fast zehnmal so stark ausfiel. Anders als 2008, das für Deutschland vor allem eine importierte Bankenkrise ohne eigene Immobilienblase war, wirkte die Zinswende 2022 direkt: Die EZB erhöhte die Leitzinsen gezielt gegen die Inflation, wodurch Bauzinsen von rund 1 % auf 3,5 bis 4 % stiegen und Finanzierung unmittelbar verteuerten, statt wie 2008 durch Zinssenkungen abgefedert zu werden.

Die Preisschere zwischen Städten aus H2 vergrößerte sich real, nicht nur durch das allgemeine Preisniveau. München und Hamburg zogen deutlich davon, während Städte wie Chemnitz kaum oder gar nicht profitierten. Bei der Erschwinglichkeit zeigte sich ein differenzierteres Bild: Frankfurt am Main lag hier vorn, nicht München, was zeigt, dass ein hohes Ausgangspreisniveau ebenso stark wirkt wie ein starker Preisanstieg.

Aus H3 ging neben dem bestätigten stärkeren Wachstum von apartment ein eigenständiger Befund hervor: multi_family (Mehrfamilienhäuser) verhielt sich als investorengetriebenes Marktsegment deutlich volatiler als selbstgenutzte Wohnformen, mit dem stärksten Einzelanstieg und dem stärksten Einbruch aller drei Immobilientypen.

Die Erschwinglichkeit lieferte schließlich die wichtigste ergänzende Erkenntnis der Analyse: Sinkende Kaufpreise ab 2023 bedeuteten keine automatische Entlastung für Käufer, da gestiegene Finanzierungskosten den Effekt überlagerten, genau die Zinsdynamik, die auch den Krisenvergleich erklärt. Wohnen blieb 2024 spürbar weniger erschwinglich als vor der Boomphase, obwohl die Preise selbst bereits wieder zurückgingen.

Insgesamt zeigt die Analyse, dass einzelne Kennzahlen wie der Durchschnittspreis für sich genommen ein irreführendes Bild liefern können. Erst das Zusammenspiel aus Zeitverlauf, regionaler Streuung, Immobilientyp und Finanzierungskosten ergibt ein belastbares Verständnis der tatsächlichen Marktdynamik.